# Parse Email

Parse mbox-format `.txt` files in `sample-data/` into individual emails.

Each file is a thread: messages separated by mbox `From ` envelope lines, each a standard RFC-822 message (`From:`, `To:`, `Date:`, `Subject:` headers + body).


In [2]:
import mailbox
import re
from email.utils import getaddresses
from pathlib import Path

SAMPLE_DIR = Path("..") / "sample-data"

In [3]:
def addrs(header):
    """Extract sorted, lowercased email addresses from a To/From header."""
    if not header:
        return []
    return sorted(a.lower() for _, a in getaddresses([header]) if a)


def normalize(text):
    """Collapse whitespace so near-duplicates compare equal."""
    return re.sub(r"\s+", " ", text).strip()


def parse_file(path):
    """Parse one mbox .txt thread into a list of email dicts."""
    emails = []
    for order, msg in enumerate(mailbox.mbox(str(path))):
        body = msg.get_payload()
        emails.append(
            {
                "doc_id": path.name,
                "canon_order": order,
                "from": addrs(msg["From"]),
                "to": addrs(msg["To"]),
                "subject": msg["Subject"],
                "date": msg["Date"],
                "content": normalize(body),
            }
        )
    return emails

In [4]:
all_emails = []
for path in sorted(SAMPLE_DIR.glob("*.txt")):
    emails = parse_file(path)
    all_emails.extend(emails)
    print(f"{path.name}: {len(emails)} emails")

print(f"\ntotal: {len(all_emails)} emails")

doc1.txt: 1 emails
doc2.txt: 2 emails
doc3.txt: 2 emails
doc4.txt: 3 emails
doc5.txt: 3 emails

total: 11 emails


In [5]:
for e in all_emails:
    print(f"[{e['doc_id']} #{e['canon_order']}] {e['from']} -> {e['to']}")
    print(f"  subject: {e['subject']}")
    print(f"  content: {e['content']}")
    print()

[doc1.txt #0] ['alice@example.com'] -> ['bob@example.com', 'carol@example.com']
  subject: Project kickoff
  content: Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice

[doc2.txt #0] ['alice@example.com'] -> ['bob@example.com', 'carol@example.com']
  subject: Project kickoff
  content: Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice

[doc2.txt #1] ['bob@example.com'] -> ['alice@example.com', 'carol@example.com']
  subject: Re: Project kickoff
  content: Hi Alice, Tuesday works great for me. Let's say 10am? Bob

[doc3.txt #0] ['alice@example.com'] -> ['bob@example.com', 'carol@example.com']
  subject: Project kickoff
  content: Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice

[doc3.txt #1] ['bob@example.com'] -> ['alice@example.com', 'carol@example.com']
  subject: Re: Project kickoff
  content: Hi Alice, Tuesday works great for me. Let's say 10am? Bob

[doc4.tx

In [6]:
all_emails

[{'doc_id': 'doc1.txt',
  'canon_order': 0,
  'from': ['alice@example.com'],
  'to': ['bob@example.com', 'carol@example.com'],
  'subject': 'Project kickoff',
  'date': 'Mon, 1 Jan 2024 09:00:00 +0000',
  'content': "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice"},
 {'doc_id': 'doc2.txt',
  'canon_order': 0,
  'from': ['alice@example.com'],
  'to': ['bob@example.com', 'carol@example.com'],
  'subject': 'Project kickoff',
  'date': 'Mon, 1 Jan 2024 09:00:00 +0000',
  'content': "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice"},
 {'doc_id': 'doc2.txt',
  'canon_order': 1,
  'from': ['bob@example.com'],
  'to': ['alice@example.com', 'carol@example.com'],
  'subject': 'Re: Project kickoff',
  'date': 'Mon, 1 Jan 2024 10:00:00 +0000',
  'content': "Hi Alice, Tuesday works great for me. Let's say 10am? Bob"},
 {'doc_id': 'doc3.txt',
  'canon_order': 0,
  'from': ['alice@example.com'],
  'to': ['bob@examp